In [0]:

#import functions
from pyspark.sql.functions import col, to_date, upper, current_timestamp

In [0]:
#converting bronze claims table to silver
df = spark.table('insurance_db.bronze.claims')
df = df.withColumn('claim_id', col('policy_id').cast('int'))
df = df.withColumn('policy_id', col('policy_id').cast('int'))
df = df.withColumn('claim_amount', col('claim_amount').cast('double'))
df = df.withColumn('claim_date', to_date(col('claim_date')))
df = df.withColumn('status', upper(col('status')))
df = df.dropna(subset=['policy_id'])
df = df.withColumn('processed_at', current_timestamp())

In [0]:
#update silver claims table
df.write.format('delta').mode('overwrite').saveAsTable('insurance_db.silver.claims')

In [0]:
#converting bronze policies table to silver
df = spark.table('insurance_db.bronze.policies')
df = df.withColumn('policy_id', col('policy_id').cast('int'))
df = df.withColumn('premium', col('claim_amount').cast('double'))
df = df.withColumn('start_date', to_date(col('start_date')))
df = df.withColumn('status', upper(col('status')))
df = df.withColumn('processed_at', current_timestamp())

In [0]:
#update silver policies table
df.write.format('delta').mode('overwrite').saveAsTable('insurance_db.silver.policies')

In [0]:
#show DISTINCT status in silver claims table
spark.sql('SELECT DISTINCT status FROM insurance_db.silver.claims').show()

+--------+
|  status|
+--------+
|APPROVED|
| PENDING|
|  DENIED|
+--------+



In [0]:
#show DISTINCT status in silver policies table
spark.sql('SELECT DISTINCT status FROM insurance_db.silver.policies').show()

+--------+
|  status|
+--------+
|  ACTIVE|
|INACTIVE|
+--------+

